In [14]:
import os, numpy as np
from PIL import Image
import torch, torch.nn as nn
import torch.nn.functional as F
import timm

torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)


Device: cuda


In [15]:
# 1×1 decoder head (same as training)
class OneLayerDecoder(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, 1, kernel_size=1)  # -> 1 logit channel
    def forward(self, feat, out_hw):
        # feat: [B,C,h,w]  out_hw: (H,W) = input size
        logits_low = self.proj(feat)                    # [B,1,h,w]
        return F.interpolate(logits_low, size=out_hw, mode='bilinear', align_corners=False)

# Encoder + head wrapper (same structure as in the training script)
class EncoderHead(nn.Module):
    def __init__(self, backbone_name="mobilenetv3_small_050.lamb_in1k"):
        super().__init__()
        # ask timm for the last spatial feature map
        self.encoder = timm.create_model(backbone_name, pretrained=True,
                                         features_only=True, out_indices=(-1,))
        in_ch = self.encoder.feature_info[-1]['num_chs']
        self.head = OneLayerDecoder(in_ch)
    def forward(self, x):
        feat = self.encoder(x)[0]        # [B,C,h,w]
        H, W = x.shape[-2:]
        return self.head(feat, (H, W))   # [B,1,H,W]


In [17]:
# Point to YOUR best checkpoint (from your screenshot folder)
BACKBONE   = "mobilenetv3_small_050.lamb_in1k"
CHECKPOINT = "/zpool/vladlab/active_drive/omaltz/scripts/geogaze/weights_bc_gc_left/mobilenetv3_small_050.lamb_in1k_maskL_bc_gc_best_1.pth.tar"

assert os.path.isfile(CHECKPOINT), f"Checkpoint not found: {CHECKPOINT}"

# Build model and load weights
model = EncoderHead(BACKBONE).to(device)
ckpt  = torch.load(CHECKPOINT, map_location='cpu')

# Most of our training scripts saved under 'state_dict'; if not, fall back to whole ckpt
state = ckpt.get('state_dict', ckpt)
missing, unexpected = model.load_state_dict(state, strict=False)
print("Loaded. Missing keys:", len(missing), "Unexpected keys:", len(unexpected))
print("Epoch:", ckpt.get('epoch', '?'), "Best Dice:", ckpt.get('best_metric', '?'))
model.eval()


Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Loaded. Missing keys: 0 Unexpected keys: 0
Epoch: 14 Best Dice: 0.9951342940330505


EncoderHead(
  (encoder): MobileNetV3Features(
    (conv_stem): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act1): Hardswish()
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (bn1): BatchNormAct2d(
            16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): ReLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): ReLU(inplace=True)
            (conv_expand): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
            (gate): Hardsigmoid()
          )
          (conv_pw): Conv2d(16, 8, kernel_size=(1, 1), strid

In [22]:
with torch.no_grad():
    y = model(torch.randn(1,3,224,224, device=device))
print(y.shape)  # expect: torch.Size([1, 1, 224, 224])


torch.Size([1, 1, 224, 224])


In [28]:
# === Run images through the loaded model and save outputs ===
import os, glob, numpy as np
from PIL import Image
import torch

# ---- EDIT THESE PATHS ONLY ----
IMAGE_DIR = "/zpool/vladlab/active_drive/omaltz/scripts/geogaze/stimuli/out/test stimuli"
OUT_DIR   = "/zpool/vladlab/active_drive/omaltz/scripts/geogaze/stimuli/out/test outputs"
os.makedirs(OUT_DIR, exist_ok=True)
# --------------------------------

# ImageNet normalization (in case not already defined)
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def load_rgb(p):
    return Image.open(p).convert("RGB")

def to_tensor_norm(pil_img):
    arr = np.asarray(pil_img, dtype=np.float32) / 255.0   # HWC
    t = torch.from_numpy(arr).permute(2,0,1)              # CHW
    return (t - IMAGENET_MEAN) / IMAGENET_STD

# collect PNGs
paths = sorted(p for p in glob.glob(os.path.join(IMAGE_DIR, "*.png")) if os.path.isfile(p))
print(f"Found {len(paths)} PNGs in {IMAGE_DIR}")

# run inference and save outputs
model.eval()
with torch.no_grad():
    for p in paths:
        img = load_rgb(p)
        x = to_tensor_norm(img).unsqueeze(0).to(device, non_blocking=True)

        logits  = model(x)                         # [1,1,H,W]
        bg_prob = torch.sigmoid(logits)[0,0].cpu().numpy()   # background probability in [0,1]
        fg_prob = 1.0 - bg_prob                               # foreground probability in [0,1]

        # Save a raw probability heatmap (no threshold). 16-bit preserves detail.
        stem = os.path.splitext(os.path.basename(p))[0]
        Image.fromarray((np.clip(fg_prob,0,1)*65535).astype(np.uint16)).save(
            os.path.join(OUT_DIR, f"{stem}_fgprob.png")
        )

        THRESHOLD = 0.20     
        mask_u8 = (fg_prob > THRESHOLD).astype(np.uint8) * 255  # object = white
        Image.fromarray(mask_u8).save(os.path.join(OUT_DIR, f"{stem}_mask.png"))

print(f"Done. Saved outputs to: {OUT_DIR}")


Found 8 PNGs in /zpool/vladlab/active_drive/omaltz/scripts/geogaze/stimuli/out/test stimuli
Done. Saved outputs to: /zpool/vladlab/active_drive/omaltz/scripts/geogaze/stimuli/out/test outputs
